# Solar filament segmentation: the whole inference pipeline

This notebook turns the 180 test H-alpha frames into the submission CSV with the weights stored in `weights/`.
It runs the same six stages, with the same code and file layout, as the runs that produced our leaderboard
submission, and reproduces that CSV byte for byte (checked on Kaggle's T4 GPUs).

| stage | what it does | time on 2× T4 |
|---|---|---|
| 1. preprocess | limb fit, limb-darkening flat field, radius channel | 1 min |
| 2. dense maps | U-Net + UNet++ (ResNet34), 5 folds each, 512-px tiles, 8 flips/rotations | 111 min |
| 3. detections | YOLO11-seg small @1024 and medium @1280, 5 folds each | 18 min |
| 4. candidate bank | 15 nested level sets of the averaged map + detector-carved pieces, 58 descriptors each | 7 min |
| 5. selection | hurdle LightGBM scores and an exact tree search that maximises pooled PQ | 6 min |
| 6. refiner | each kept filament redrawn in a rotated native-scale crop, pasted back | 8 min |
| total | | 151 min |

Set `DATA_ROOT` to the competition's `MAGFiLO_1.0_Kaggle_2026` folder. Each stage skips work whose output
already exists, so an interrupted run resumes where it stopped.

In [ ]:
# Install the pinned packages once (see requirements.txt for the torch/CUDA note).
# %pip install -r requirements.txt

In [ ]:
import os, sys, time
from pathlib import Path

DATA_ROOT = Path(os.environ.get("FILAMENT_DATA_ROOT",  # your copy of the competition data, else Kaggle's
                               "/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"))
REPO = Path.cwd()
WORK = REPO / "work"            # intermediate products: cache, maps, detections, bank, crops
OUT = REPO / "submission.csv"

os.environ["FILAMENT_DATA_ROOT"] = str(DATA_ROOT)
sys.path.insert(0, str(REPO / "pipeline"))
from infer import Run

run = Run(weights=REPO / "weights", work=WORK)
print(run.cfg)

## 1. Preprocessing
The solar limb is fitted robustly (a plain threshold overshoots it because of the bright halo), the image is divided
by a fitted limb-darkening profile, and a radial-distance channel is added. Code: `pipeline/preprocess.py` and
`pipeline/filcore/limb.py` (our implementation of filament-seg's preprocessing, which the networks were trained on).

In [ ]:
run.preprocess()

## 2. Dense probability maps
Ten networks (U-Net and UNet++ with a ResNet34 encoder, one per cross-validation fold) predict a filament probability
for every pixel; the fold models are averaged. Weights are stored in fp16 and give the same logits as the fp32
originals under fp16 autocast. With two GPUs, the checkpoints are split between them.

In [ ]:
run.dense()

## 3. Instance detections
Two YOLO11 instance-segmentation models per fold provide instance masks (down to confidence 0.05); they are used to
carve candidates and as evidence for the scorer.

In [ ]:
run.detect()

## 4. Candidate bank
For each frame the averaged map is thresholded at 15 levels; the connected components form a tree (a component at a
higher threshold lies inside one at a lower threshold). Detector masks carve extra pieces. Every candidate gets 58
descriptors (shape, map evidence inside and around it, model disagreement, detector agreement, tree position).

In [ ]:
run.bank()

## 5. Learned selection
A hurdle model (LightGBM) predicts each candidate's expected IoU per annotator view, u-hat. Pooled PQ is maximised by
keeping the disjoint set with the largest sum of (u-hat − τ), τ = 0.24 (half the achievable PQ); an exact dynamic
program over each tree finds it. This also writes `work/submission_stage1.csv`, the selection before refinement.

In [ ]:
run.select()

## 6. Refiner and submission
Each kept filament is cropped along its own axis at native resolution; a UNet++ (5 folds × 4 flips) redraws it, and the
prediction is pasted back at threshold 0.5, its parts joined into one region, overlaps resolved by score.

In [ ]:
run.refine(OUT)
print({k: round(v, 1) for k, v in run.minutes.items()}, "minutes")

## Check and look at the result

In [ ]:
import pandas as pd
sub = pd.read_csv(OUT)
print(len(sub), "instances over", sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "test frames")
sub.head()

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from filcore.rle import decode, read_csv
from filcore.limb import load_gray

preds = read_csv(OUT)
stems = sorted(preds)[:2]
fig, axes = plt.subplots(1, len(stems), figsize=(12, 6))
for ax, stem in zip(np.atleast_1d(axes), stems):
    img = load_gray(next((DATA_ROOT / "test" / "test_images").glob(f"{stem}.*"))).astype(float)
    ax.imshow(img, cmap="gray")
    for k, rle in enumerate(preds[stem]):
        m = decode(rle)
        ax.contour(m, levels=[0.5], colors=[["#2a78d6", "#eb6834", "#1baf7a"][k % 3]], linewidths=0.8)
    ax.set_title(f"{stem}: {len(preds[stem])} filaments"); ax.axis("off")
plt.tight_layout()

## Validation and training
All numbers in the report are out of fold over five date-grouped folds of the 707 training observations
(`pipeline/cv5_folds.json`): fixed fusion 0.4232 → learned selection 0.4417 → with the refiner 0.4475 PQ.
Training code: `pipeline/train_cv.py` (dense models), `pipeline/detector.py` (detectors),
`pipeline/bank.py` + `pipeline/scorer.py` / `submit.py --save-model` (scorer), `pipeline/refine.py train` (refiner);
`TRAINING.md` lists the exact commands.